In [ ]:
"""
add_more_plates_to_gel.py

Adds rigid graphene-like plates to ALL SIX faces of an isolated gel data file:
  - Top plate    (+z face) : xy-plane lattice, spanning full box in xy
  - Bottom plate (-z face) : xy-plane lattice, spanning full box in xy
  - Front plate  (+y face) : xz-plane lattice, spanning full box in x and z, placed at yhi
  - Back plate   (-y face) : xz-plane lattice, spanning full box in x and z, placed at ylo
  - Right plate  (+x face) : yz-plane lattice, spanning full box in y and z, placed at xhi
  - Left plate   (-x face) : yz-plane lattice, spanning full box in y and z, placed at xlo

Plates are placed exactly at the simulation box boundaries, NOT offset from the
polymer surface. This prevents polymer chains from crossing outside the plates.
Solvent atoms (type 3) are never deleted — plates do not interact with solvent.

Design
------
- Plate atoms are atom type 4 (mass 1.0), arranged on a square lattice.
- NO bonds are created between plate atoms and the polymer network.
  Contact is purely through WCA (LJ) interactions. This avoids lateral
  forces on surface chains when the plates compress the gel — bonded
  plate atoms would stretch sideways as the plate moves in z, transmitting
  unphysical shear to the network edges.
- Output data file has 4 atom types and 1 bond type (FENE only, unchanged).

Plate-Plate Interactions
------------------------
Plate-plate interactions MUST be disabled in the LAMMPS input script.
Recommended approach (add to LAMMPS input):
    neigh_modify exclude type 4 4
Alternative (zero out epsilon):
    pair_coeff 4 4 0.0 1.0 0.0

Bond style needed in LAMMPS input (unchanged from gel):
    bond_style fene
    bond_coeff 1 30.0 1.5 1.0 1.0   # polymer-polymer (FENE)

Pair coeffs for plate (type 4) — add to LAMMPS input:
    pair_coeff 4 4 0.0 1.0 0.0      # plate-plate  DISABLED
    pair_coeff 1 4 1.0 1.0 1.122    # polymer-plate (WCA)
    pair_coeff 2 4 1.0 1.0 1.122    # crosslinker-plate (WCA)
    pair_coeff 3 4 0.0 1.0 0.0      # solvent-plate DISABLED
"""


import numpy as np

# ── Defaults ─────────────────────────────────────────────────────────────────
PLATE_TYPE      = 4      # new atom type for plate beads
# PLATE_SPACING was 0.2, which is 5x finer than the bead diameter and made the
# six plates 1,122,406 atoms = 77% of a 1.46M-atom simulation.  The 2026-07-26
# Pod run spent 0.35% of its wall time on Pair+Bond and 98% on Neigh+Comm+
# imbalance, managing 1.215 timesteps/s -> ~8.4 days for the full script against
# a 24 h walltime.  A WCA wall does not need 0.2 sigma: at 1.0 sigma spacing the
# worst-case gap (square centre, r = 0.707) still carries a ~225 epsilon barrier,
# i.e. completely impenetrable at T = 1.  1.0 cuts plate atoms by 25x.
PLATE_SPACING   = 1.0    # square lattice constant (σ)
# Side plates (front/back/left/right) span the polymer z-extent + this margin,
# instead of the full box z-range.  See the side_zlo/side_zhi block in
# add_six_plates() for why the old full-span version was pure dead weight.
PLATE_Z_MARGIN  = 5.0    # (σ)
# Plates are placed at box boundaries — no offset or surface detection needed.


# ── I/O helpers ──────────────────────────────────────────────────────────────

def parse_lammps_data(filename):
    """Return atoms, bonds, box_bounds, masses from a LAMMPS molecular data file."""
    atoms, bonds, masses = [], [], {}
    box = {}

    with open(filename) as f:
        lines = f.readlines()

    i = 0
    while i < len(lines):
        line = lines[i].strip()

        if 'atom types' in line:
            box['n_atom_types'] = int(line.split()[0])
        elif 'bond types' in line:
            box['n_bond_types'] = int(line.split()[0])
        elif 'xlo xhi' in line:
            p = line.split(); box['xlo'], box['xhi'] = float(p[0]), float(p[1])
        elif 'ylo yhi' in line:
            p = line.split(); box['ylo'], box['yhi'] = float(p[0]), float(p[1])
        elif 'zlo zhi' in line:
            p = line.split(); box['zlo'], box['zhi'] = float(p[0]), float(p[1])

        elif line == 'Masses':
            i += 2
            while i < len(lines) and lines[i].strip() and \
                    not lines[i].strip()[0].isalpha():
                p = lines[i].split()
                if len(p) >= 2:
                    masses[int(p[0])] = float(p[1])
                i += 1
            continue

        elif line.startswith('Atoms'):
            i += 2
            while i < len(lines) and lines[i].strip() and \
                    not lines[i].strip()[0].isalpha():
                p = lines[i].split()
                if len(p) >= 6:
                    atoms.append({
                        'id':   int(p[0]),
                        'mol':  int(p[1]),
                        'type': int(p[2]),
                        'x':    float(p[3]),
                        'y':    float(p[4]),
                        'z':    float(p[5]),
                    })
                i += 1
            continue

        elif line.startswith('Bonds'):
            i += 2
            while i < len(lines) and lines[i].strip() and \
                    not lines[i].strip()[0].isalpha():
                p = lines[i].split()
                if len(p) >= 4:
                    bonds.append({
                        'id':    int(p[0]),
                        'type':  int(p[1]),
                        'atom1': int(p[2]),
                        'atom2': int(p[3]),
                    })
                i += 1
            continue

        i += 1

    return atoms, bonds, box, masses


def write_lammps_data(filename, atoms, bonds, box, masses):
    """Write LAMMPS molecular data file. Atom IDs are renumbered 1..N."""
    old2new = {a['id']: i + 1 for i, a in enumerate(atoms)}

    valid_bonds = []
    for i, b in enumerate(bonds):
        a1 = old2new.get(b['atom1'])
        a2 = old2new.get(b['atom2'])
        if a1 and a2:
            valid_bonds.append({'id': i + 1, 'type': b['type'],
                                 'atom1': a1, 'atom2': a2})

    # Never SHRINK the declared type count below what the input declared.
    # The old code used max(type present), which silently downgraded a file that
    # declared 5 atom types but contained only types 1-4.  compress_slab.lmp then
    # aborted at setup on its
    #     neigh_modify exclude type 4 5   /   pair_coeff 5 5 ...
    # lines with an invalid-atom-type error, before a single timestep ran.
    n_types  = max(max(a['type'] for a in atoms), box.get('n_atom_types', 0))
    n_btypes = max(max((b['type'] for b in valid_bonds), default=1),
                   box.get('n_bond_types', 0))

    with open(filename, 'w') as f:
        f.write("LAMMPS data file - isolated gel with six shear plates\n\n")
        f.write(f"{len(atoms)} atoms\n")
        f.write(f"{len(valid_bonds)} bonds\n\n")
        f.write(f"{n_types} atom types\n")
        f.write(f"{n_btypes} bond types\n\n")
        f.write(f"{box['xlo']:.6f} {box['xhi']:.6f} xlo xhi\n")
        f.write(f"{box['ylo']:.6f} {box['yhi']:.6f} ylo yhi\n")
        f.write(f"{box['zlo']:.6f} {box['zhi']:.6f} zlo zhi\n\n")

        f.write("Masses\n\n")
        for t in range(1, n_types + 1):
            f.write(f"{t} {masses.get(t, 1.0):.4f}\n")

        f.write("\nAtoms\n\n")
        for i, a in enumerate(atoms, 1):
            f.write(f"{i} {a['mol']} {a['type']} "
                    f"{a['x']:.6f} {a['y']:.6f} {a['z']:.6f}\n")

        if valid_bonds:
            f.write("\nBonds\n\n")
            for b in valid_bonds:
                f.write(f"{b['id']} {b['type']} {b['atom1']} {b['atom2']}\n")


# ── Plate generation ──────────────────────────────────────────────────────────

def make_plate_xy(box, z_plane, spacing, mol_id):
    """
    Top or bottom plate: square lattice in the xy-plane at z = z_plane.
    Spans [xlo, xhi) x [ylo, yhi) — tiles perfectly under PBC.
    """
    lx = box['xhi'] - box['xlo']
    ly = box['yhi'] - box['ylo']

    nx = int(np.floor(lx / spacing))
    ny = int(np.floor(ly / spacing))

    dx = lx / nx
    dy = ly / ny
    x0 = box['xlo'] + dx / 2.0
    y0 = box['ylo'] + dy / 2.0

    plate = []
    for ix in range(nx):
        for iy in range(ny):
            plate.append({
                'id':   None,
                'mol':  mol_id,
                'type': PLATE_TYPE,
                'x':    x0 + ix * dx,
                'y':    y0 + iy * dy,
                'z':    z_plane,
            })
    return plate


def make_plate_xz(box, y_plane, z_lo, z_hi, spacing, mol_id):
    """
    Front (+y) or back (-y) plate: square lattice in the xz-plane at y = y_plane.
    Spans [xlo, xhi) in x (tiles under PBC) and [z_lo, z_hi] in z (full box z-extent).
    """
    lx = box['xhi'] - box['xlo']
    lz = z_hi - z_lo

    nx = int(np.floor(lx / spacing))
    nz = max(1, int(np.round(lz / spacing)))

    dx = lx / nx
    dz = lz / nz if nz > 1 else spacing
    x0 = box['xlo'] + dx / 2.0
    z0 = z_lo + dz / 2.0

    plate = []
    for ix in range(nx):
        for iz in range(nz):
            plate.append({
                'id':   None,
                'mol':  mol_id,
                'type': PLATE_TYPE,
                'x':    x0 + ix * dx,
                'y':    y_plane,
                'z':    z0 + iz * dz,
            })
    return plate


def make_plate_yz(box, x_plane, z_lo, z_hi, spacing, mol_id):
    """
    Right (+x) or left (-x) plate: square lattice in the yz-plane at x = x_plane.
    Spans [ylo, yhi) in y (tiles under PBC) and [z_lo, z_hi] in z (full box z-extent).
    """
    ly = box['yhi'] - box['ylo']
    lz = z_hi - z_lo

    ny = int(np.floor(ly / spacing))
    nz = max(1, int(np.round(lz / spacing)))

    dy = ly / ny
    dz = lz / nz if nz > 1 else spacing
    y0 = box['ylo'] + dy / 2.0
    z0 = z_lo + dz / 2.0

    plate = []
    for iy in range(ny):
        for iz in range(nz):
            plate.append({
                'id':   None,
                'mol':  mol_id,
                'type': PLATE_TYPE,
                'x':    x_plane,
                'y':    y0 + iy * dy,
                'z':    z0 + iz * dz,
            })
    return plate


# ── Main ─────────────────────────────────────────────────────────────────────

def add_six_plates(input_file, output_file,
                   spacing=PLATE_SPACING,
                   z_margin=PLATE_Z_MARGIN,
                   drop_types=None):
    """
    drop_types : iterable of atom types to delete outright, e.g. {5} or {4, 5}.
        The leftover single-axis piston/support atoms from the pre-plate
        equilibration are NOT removed by default (this script has never removed
        them - it only deletes polymer near the boundary).  They are inert during
        the run (no integrator acts on them: they are outside `group gel`, which
        is types 1 2 3, and outside the six mol-ID plate groups), but they are a
        RIGID INCLUSION inside the material whose bulk modulus you are measuring,
        which biases K upward.  The 3 sigma exclusion region in compress_slab.lmp
        masks the six faces only, not the piston.  Removing them is a science
        decision - hence opt-in, not default.
    """

    print("=" * 60)
    print(f"add_more_plates_to_gel.py")
    print(f"  Input  : {input_file}")
    print(f"  Output : {output_file}")
    print(f"  Lattice spacing : {spacing} σ")
    print(f"  Plate placement : AT BOX BOUNDARIES (no gel surface offset)")
    print(f"  Bond mode       : NONE (pure WCA contact only)")
    print(f"  Solvent (type 3): never deleted, plate-solvent interaction = 0")
    print(f"  Plates: top (+z), bottom (-z), front (+y), back (-y), right (+x), left (-x)")
    print("=" * 60)

    atoms, bonds, box, masses = parse_lammps_data(input_file)
    print(f"Read {len(atoms)} atoms, {len(bonds)} bonds")

    if drop_types:
        drop_types = set(drop_types)
        drop_ids = {a['id'] for a in atoms if a['type'] in drop_types}
        atoms = [a for a in atoms if a['id'] not in drop_ids]
        bonds = [b for b in bonds
                 if b['atom1'] not in drop_ids and b['atom2'] not in drop_ids]
        print(f"drop_types={sorted(drop_types)}: removed {len(drop_ids)} atoms. "
              f"Remaining: {len(atoms)}")
        print("  (declared atom-type count is preserved, so compress_slab.lmp's "
              "type-5 pair_coeff lines stay valid)")

    _POLY_TYPES = {1, 2}

    # ── Compute plate positions FIRST (needed for the clearance cut below) ────
    # Plates sit exactly on the box faces; no offset from gel surface.
    # Side plates span the FULL box z-extent so chains cannot cross.
    # LAMMPS PBC convention: primary cell is [lo, hi).
    # Atoms placed at exactly xhi/yhi/zhi get wrapped back to xlo/ylo/zlo,
    # which would make the 'hi' plate coincide with the 'lo' plate.
    # Fix: shift hi-face plates one spacing inward so they stay in [lo, hi).
    z_top_plate   = box['zhi'] - spacing  # one spacing inside → won't wrap to zlo
    z_bot_plate   = box['zlo']
    y_front_plate = box['yhi'] - spacing  # one spacing inside → won't wrap to ylo
    y_back_plate  = box['ylo']
    x_right_plate = box['xhi'] - spacing  # one spacing inside → won't wrap to xlo
    x_left_plate  = box['xlo']

    # ── Delete polymer atoms that overlap with the ACTUAL plate planes ────────
    # FIX (2026-07-31): clearance is now measured from each plate's real
    # coordinate (x_right_plate, y_front_plate, ... ), not from the raw box
    # boundary.  The three "hi"-face plates (top/front/right) sit ONE
    # SPACING INSIDE the box edge (see above), but the old cut measured
    # PLATE_CLEAR inward from the raw boundary.  With PLATE_SPACING ==
    # PLATE_CLEAR == 1.0 those two happened to cancel exactly, so the
    # deletion line and the front/right plate plane landed on the same
    # coordinate — surviving polymer got ZERO clearance from those two
    # plates (measured overlaps down to 0.05 sigma) instead of the intended
    # 1 sigma, and blew up on the very first NVT step.  The "lo"-face plates
    # (bottom/back/left) sit exactly at the box edge with no inward shift,
    # so they were never affected — hence the deletion. Measuring from the
    # real plate coordinate makes all six faces symmetric regardless of the
    # inward shift.
    PLATE_CLEAR = 1

    def _at_boundary(a):
        return (a['x'] <= x_left_plate + PLATE_CLEAR or a['x'] >= x_right_plate - PLATE_CLEAR or
                a['y'] <= y_back_plate + PLATE_CLEAR or a['y'] >= y_front_plate - PLATE_CLEAR or
                a['z'] <= z_bot_plate + PLATE_CLEAR or a['z'] >= z_top_plate - PLATE_CLEAR)

    del_ids = {a['id'] for a in atoms
               if a['type'] in _POLY_TYPES and _at_boundary(a)}
    n_del = len(del_ids)
    if n_del:
        atoms = [a for a in atoms if a['id'] not in del_ids]
        bonds = [b for b in bonds
                 if b['atom1'] not in del_ids and b['atom2'] not in del_ids]
        print(f"Removed {n_del} polymer atoms overlapping with plate planes "
              f"(+ their bonds). Remaining: {len(atoms)} atoms, {len(bonds)} bonds")
    else:
        print("No polymer atoms at plate planes — nothing deleted")

    print(f"\nBox boundaries:")
    print(f"  x: {box['xlo']:.3f} → {box['xhi']:.3f}")
    print(f"  y: {box['ylo']:.3f} → {box['yhi']:.3f}")
    print(f"  z: {box['zlo']:.3f} → {box['zhi']:.3f}")

    # -- Side-plate z-extent --------------------------------------------------
    # FIX (2026-07-31): side plates now span the FULL plate-to-plate range,
    # z_bot_plate -> z_top_plate, not just the current polymer footprint +/-
    # z_margin.  The margin-based version left the region between the side
    # plate's edge and the top/bottom plate completely unwalled.  Since the
    # top/bottom plates only ever move INWARD during compression, the gel's
    # z-extent grows toward z_bot_plate/z_top_plate over the course of the
    # run -- side walls that stop short of that range let polymer squeeze out
    # sideways through the corners once compression pushes the gel past the
    # old margin.  That was the root cause of the FENE-too-long warnings and
    # "Bond atoms missing" crash on the 2026-07-31 Pod run (together with the
    # PLATE_TYPE collision fixed via drop_types, below).
    # z_margin is kept as a parameter for compatibility but no longer shrinks
    # the span; full coverage costs ~25% more side-plate atoms at
    # PLATE_SPACING=1.0, nowhere near the 0.2-spacing blowup this was
    # originally written to avoid.
    side_zlo, side_zhi = z_bot_plate, z_top_plate
    print(f"\nSide-plate z-span (full plate-to-plate coverage): "
          f"{side_zlo:.3f} -> {side_zhi:.3f}  "
          f"({side_zhi - side_zlo:.1f} of {box['zhi'] - box['zlo']:.1f} sigma box)")

    # -- Plate molecule IDs: number from the GEL max, not the overall max ------
    # compress_slab.lmp resolves the six plates with
    #     group   gel type 1 2 3
    #     compute max_mol_gel gel reduce max v_mol_id
    #     group   plate_top molecule (max_mol_gel + 1)
    # i.e. the max over GEL types only.  This script used to take the max over
    # ALL atoms, so if leftover piston/support atoms (types 4/5) carried higher
    # mol IDs than the gel, the plates were stamped ABOVE where the .lmp looks:
    # every plate_* group comes back EMPTY, `compute com` divides by zero mass,
    # gap_z_0 and compress_strain_vol go NaN, `fix halt` never fires (NaN >=
    # target is false), the ramp runs its full 30000-step cap at NaN velocity,
    # and atoms are lost.  Match the .lmp exactly.
    _GEL_TYPES = {1, 2, 3}
    gel_mols = [a['mol'] for a in atoms if a['type'] in _GEL_TYPES]
    max_mol     = max(gel_mols) if gel_mols else max(a['mol'] for a in atoms)
    max_mol_all = max(a['mol'] for a in atoms)
    if max_mol_all > max_mol:
        print(f"  NOTE: non-gel atoms reach mol {max_mol_all}, above the gel max "
              f"{max_mol}.")
        print(f"        Plates numbered {max_mol + 1}..{max_mol + 6} to match "
              f"compress_slab.lmp's compute max_mol_gel.")
    clash = sorted({a['mol'] for a in atoms if max_mol < a['mol'] <= max_mol + 6})
    if clash:
        print(f"  *** WARNING: non-gel atoms already occupy mol IDs {clash}.")
        print( "  *** They will be swept into the plate_* groups in "
               "compress_slab.lmp and driven by fix move.")
        print( "  *** Fix: rerun with drop_types={4, 5}, or renumber them.")

    plate_top   = make_plate_xy(box, z_top_plate,   spacing, mol_id=max_mol + 1)
    plate_bot   = make_plate_xy(box, z_bot_plate,   spacing, mol_id=max_mol + 2)
    plate_front = make_plate_xz(box, y_front_plate,
                                 side_zlo, side_zhi, spacing, mol_id=max_mol + 3)
    plate_back  = make_plate_xz(box, y_back_plate,
                                 side_zlo, side_zhi, spacing, mol_id=max_mol + 4)
    plate_right = make_plate_yz(box, x_right_plate,
                                 side_zlo, side_zhi, spacing, mol_id=max_mol + 5)
    plate_left  = make_plate_yz(box, x_left_plate,
                                 side_zlo, side_zhi, spacing, mol_id=max_mol + 6)

    print(f"\nPlate positions and sizes:")
    print(f"  Top plate   (+z): z = {z_top_plate:.3f}  |  {len(plate_top)} atoms")
    print(f"  Bottom plate(-z): z = {z_bot_plate:.3f}  |  {len(plate_bot)} atoms")
    print(f"  Front plate (+y): y = {y_front_plate:.3f}  |  {len(plate_front)} atoms")
    print(f"  Back plate  (-y): y = {y_back_plate:.3f}  |  {len(plate_back)} atoms")
    print(f"  Right plate (+x): x = {x_right_plate:.3f}  |  {len(plate_right)} atoms")
    print(f"  Left plate  (-x): x = {x_left_plate:.3f}  |  {len(plate_left)} atoms")

    # -- Atom-count budget ----------------------------------------------------
    # The 2026-07-26 Pod run was 77% plate atoms (1,122,406 of 1,455,874) and
    # spent 0.35% of wall time on Pair+Bond vs 98% on Neigh+Comm+imbalance,
    # managing 1.215 timesteps/s -> ~8.4 days for an 880k-step script under a
    # 24 h walltime.  Print the budget so that can never happen silently again.
    n_plate_total = (len(plate_top) + len(plate_bot) + len(plate_front)
                     + len(plate_back) + len(plate_right) + len(plate_left))
    n_gel = len(atoms)
    frac = 100.0 * n_plate_total / max(1, n_plate_total + n_gel)
    print(f"\n  TOTAL plate atoms : {n_plate_total:,}  ({frac:.1f}% of final system)")
    print(f"  Gel atoms         : {n_gel:,}")
    print(f"  Final system      : {n_plate_total + n_gel:,}")
    if frac > 40.0:
        print("  *** WARNING: plates dominate the atom count.  This is what made")
        print("  ***          the 2026-07-26 run spend 98% of its time in")
        print("  ***          Neigh+Comm and miss its walltime.  Raise")
        print("  ***          PLATE_SPACING or lower PLATE_Z_MARGIN.")

    # ── Assign IDs ────────────────────────────────────────────────────────────
    next_id = max(a['id'] for a in atoms) + 1
    for a in (plate_top + plate_bot + plate_front + plate_back
              + plate_right + plate_left):
        a['id'] = next_id
        next_id += 1

    masses[PLATE_TYPE] = 1.0

    # ── Box is unchanged — plates sit exactly at existing boundaries ─────────
    new_box = dict(box)

    # ── Assemble and write ────────────────────────────────────────────────────
    all_atoms = (atoms + plate_top + plate_bot + plate_front
                 + plate_back + plate_right + plate_left)

    write_lammps_data(output_file, all_atoms, bonds, new_box, masses)

    n_plate = (len(plate_top) + len(plate_bot) + len(plate_front)
               + len(plate_back) + len(plate_right) + len(plate_left))
    print(f"\nWrote {len(all_atoms)} atoms ({n_plate} plate atoms), "
          f"{len(bonds)} bonds (unchanged) to:\n  {output_file}")
    print("=" * 60)

    # ── Summary for LAMMPS input script ──────────────────────────────────────
    print("\n── LAMMPS input script hints ────────────────────────────────")
    lx = new_box['xhi'] - new_box['xlo']
    ly = new_box['yhi'] - new_box['ylo']
    lz = new_box['zhi'] - new_box['zlo']
    print(f"  box x-size : {lx:.2f} σ  (plates at x = {x_left_plate:.3f} and {x_right_plate:.3f})")
    print(f"  box y-size : {ly:.2f} σ  (plates at y = {y_back_plate:.3f} and {y_front_plate:.3f})")
    print(f"  box z-size : {lz:.2f} σ  (plates at z = {z_bot_plate:.3f} and {z_top_plate:.3f})")
    print()
    print("  # ── Plate-plate interactions MUST be disabled ──")
    print("  neigh_modify exclude type 4 4        # recommended")
    print("  # OR: pair_coeff 4 4 0.0 1.0 0.0    # zero epsilon")
    print()
    print("  bond_style fene")
    print("  bond_coeff 1 30.0 1.5 1.0 1.0        # polymer-polymer (unchanged)")
    print()
    print("  pair_coeff 4 4 0.0 1.0 0.0            # plate-plate     DISABLED")
    print("  pair_coeff 1 4 1.0 1.0 1.122          # polymer-plate   (WCA)")
    print("  pair_coeff 2 4 1.0 1.0 1.122          # crosslinker-plate (WCA)")
    print("  pair_coeff 3 4 0.0 1.0 0.0            # solvent-plate   DISABLED (no interaction)")
    print("─" * 60)


In [ ]:
# Inputs

input_file  = "../../lammps_data_files_local/final_config_slab_support_5beads_tall_rho04_new_01_1.0_1.2_10000000.data"
output_file = "../../lammps_data_files_local/isolated_slab_support_5beads_tall_rho04_p1.5_1.0_1.0_600002_with_six_plates.data"

add_six_plates(input_file, output_file, drop_types={4, 5})
# drop_types={4, 5}: strips the leftover single-axis piston/support atoms from
# the pre-plate equilibration BEFORE adding the six new plates.  On the
# 2026-07-28 run these were left in and one of the two leftover layers
# happened to already be typed 4 -- the same type PLATE_TYPE uses for the new
# plates -- so it silently merged into the "plate" group in compress_slab.lmp
# (group plate type 4) without being one of the six mol-ID plate subgroups
# fix move actually drives.  That gave a frozen, un-thermostatted WCA wall
# sandwiched between the real moving plate and the gel surface, which is what
# produced the FENE-too-long warnings and the eventual "Bond atoms missing"
# crash.